# Mini-Projet NLP : Transformers et Système RAG**ENSA Al Hoceima** — Filière Ingénierie des Données (2ème Année)Ce notebook présente :1. **Partie 1** : Exploration des Transformers et tâches NLP avec HuggingFace.2. **Partie 2** : Étude et implémentation d'un système RAG (Retrieval-Augmented Generation).### Bibliothèques utilisées* `transformers` : modèles pré-entraînés HuggingFace* `sentence-transformers` : embeddings de documents et requêtes* `faiss` : recherche de similarité vectorielle* `torch` : exécution des modèles deep learning* `pandas`, `numpy` : manipulation de données

# Partie 1 : Exploration des Transformers et tâches NLPLes Transformers sont des architectures de deep learning très utilisées en NLP.Ils permettent de traiter du texte en capturant les relations entre les mots grâce au mécanisme d'attention.Dans cette partie, nous utilisons des modèles pré-entraînés via HuggingFace pour réaliser plusieurs tâches NLP.

In [ ]:
# Installation des bibliothèques nécessaires!pip install datasets accelerate!pip install -q transformers==4.44.2

## 1.a — Importation des bibliothèques

In [ ]:
import torchfrom transformers import AutoTokenizer, AutoModel, pipelinedevice = 0 if torch.cuda.is_available() else -1print("GPU disponible :", torch.cuda.is_available())if torch.cuda.is_available():    print("Nom du GPU :", torch.cuda.get_device_name(0))

## 1.b — Exploration du tokenizer et du modèle TransformerUn modèle Transformer ne comprend pas directement le texte brut.Le texte passe par un **tokenizer** :`Texte brut → Tokens → Identifiants numériques → Modèle Transformer`Le tokenizer découpe le texte en mots ou sous-mots, puis les transforme en nombres appelés `input_ids`.

In [ ]:
# Modèle BERT adapté au françaismodel_name = "dbmdz/bert-base-french-europeana-cased"# Chargement du tokenizertokenizer = AutoTokenizer.from_pretrained(model_name)# Chargement du modèlemodel = AutoModel.from_pretrained(model_name)# Exemple de phrasetexte = "Les architectures Transformers ont révolutionné le traitement du langage naturel."# Encodage du texteinputs = tokenizer(texte, return_tensors="pt")print("Texte original :")print(texte)print("\nTokens générés :")print(tokenizer.tokenize(texte))print("\nIdentifiants numériques des tokens :")print(inputs["input_ids"])# Passage dans le modèleoutputs = model(**inputs)print("\nForme de la dernière couche cachée :")print(outputs.last_hidden_state.shape)

## 1.c — Utilisation des pipelines HuggingFaceHuggingFace propose une fonction `pipeline` qui permet d'utiliser facilement des modèles pré-entraînés pour différentes tâches NLP.Nous testons quatre tâches :1. Analyse de sentiment2. Classification de texte (zero-shot)3. Question Answering4. Résumé automatique

### 1.c.1 — Analyse de sentimentL'analyse de sentiment détermine l'opinion exprimée dans un texte (positif, négatif, neutre).Plus le score est proche de 1, plus le modèle est confiant.

In [ ]:
sentiment_pipeline = pipeline(    "sentiment-analysis",    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",    device=device)textes = [    "Ce film est absolument magnifique !",    "Je n'ai pas du tout aimé cette expérience.",]for texte in textes:    result = sentiment_pipeline(texte)    print(f"Texte   : {texte}")    print(f"Sentiment label : {result[0]['label']} , score : {result[0]['score']:.4f}\n")

### 1.c.2 — Classification de texte (Zero-Shot)La classification zero-shot permet de classer un texte dans des catégories choisies **sans entraîner un nouveau modèle**.Le modèle compare le texte avec chaque label et retourne les probabilités.

In [ ]:
classifier = pipeline(    "zero-shot-classification",    model="facebook/bart-large-mnli",    device=device)texte_a_classer = "L'équipe de développement a déployé une nouvelle base de données vectorielle ce matin."labels_possibles = [    "informatique",    "sport",    "économie",    "politique"]resultat_classification = classifier(    texte_a_classer,    candidate_labels=labels_possibles)print("--- Classification de Texte ---")print("Texte à classer : ",texte_a_classer)print("\nRésultats :")for label, score in zip(resultat_classification["labels"], resultat_classification["scores"]):    print(f"{label} : {score:.4f}")

### 1.c.3 — Question AnsweringLe Question Answering extrait une réponse directement depuis un contexte donné.Le modèle ne génère pas librement : il cherche la réponse dans le texte fourni.

In [ ]:
qa_pipeline = pipeline(    "question-answering",    model="deepset/xlm-roberta-base-squad2",    device=device)contexte = """L'École Nationale des Sciences Appliquées d'Al Hoceima propose une filière en Ingénierie des Données.Au cours de la deuxième année, les étudiants réalisent un mini-projet portant sur l'exploration des modèles Transformerset la mise en œuvre de systèmes de génération augmentée par récupération, appelés systèmes RAG."""question = "Quel est le sujet du mini-projet ?"reponse = qa_pipeline(    question=question,    context=contexte)print("Question :", question)print("Réponse extraite :", reponse["answer"])print("Score de confiance :", round(reponse["score"], 4))

### 1.c.4 — Résumé automatiqueLe résumé automatique produit une version courte d'un texte long, en conservant les idées principales.

In [ ]:
summarizer = pipeline(    "summarization",    model="lincoln/mbart-mlsum-automatic-summarization",    device=device)texte_long = """Le traitement du langage naturel a franchi un cap important avec l'introduction de l'architecture Transformer en 2017.Contrairement aux réseaux récurrents traditionnels, qui traitent les mots de manière séquentielle,les Transformers s'appuient sur un mécanisme d'attention. Ce mécanisme permet au modèle d'analyser les relationsentre tous les mots d'une phrase simultanément. Cette innovation a facilité l'entraînement de grands modèles de langagesur des volumes massifs de données, ouvrant la voie aux modèles modernes utilisés dans la traduction automatique,la génération de texte, le question answering et les systèmes RAG."""resume = summarizer(    texte_long,    max_length=70,    min_length=25,    do_sample=False)print("Texte original :")print(texte_long)print("\nRésumé généré :")print(resume[0]["summary_text"])

# Partie 2 : Systèmes RAG (Retrieval-Augmented Generation)## 2.1 — État de l'art### Définition des systèmes RAGLe **RAG (Retrieval-Augmented Generation)** est une approche hybride qui combine un module de **récupération d'information** (retrieval) avec un module de **génération de texte** (generation). L'idée fondamentale est d'augmenter les capacités d'un modèle de langage (LLM) en lui fournissant des documents pertinents extraits d'une base de connaissances externe avant de générer une réponse.Cette approche a été introduite par Lewis et al. (2020) dans l'article *"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"*.### Limites des LLMs sans accès à des données externesLes grands modèles de langage (GPT, LLaMA, Flan-T5, etc.) présentent plusieurs limites lorsqu'ils fonctionnent sans accès à des données externes :| Limite | Description ||---|---|| **Hallucinations** | Le modèle génère des informations plausibles mais fausses || **Connaissances figées** | Les données d'entraînement ont une date de coupure || **Absence de données spécialisées** | Le modèle ne connaît pas les documents internes d'une organisation || **Manque de traçabilité** | Impossible de vérifier la source d'une réponse |### Comparaison des approches| Critère | Fine-tuning | Prompt Engineering | RAG ||---|---|---|---|| **Principe** | Ré-entraîner le modèle sur des données spécifiques | Formuler des prompts optimisés | Récupérer des documents puis générer || **Coût** | Élevé (GPU, données annotées) | Faible | Modéré || **Mise à jour des données** | Nécessite un ré-entraînement | Limitée au contexte du prompt | Mise à jour de la base documentaire || **Traçabilité** | Faible | Faible | Forte (sources identifiables) || **Cas d'usage** | Tâches très spécifiques | Prototypage rapide | QA sur corpus, chatbots documentaires |### Variantes modernes des architectures RAG1. **Naive RAG** : Pipeline séquentiel simple (retrieve → read → generate). C'est l'approche que nous implémentons dans ce notebook.2. **Advanced RAG** : Ajoute des étapes de pré-traitement (re-ranking des documents, query rewriting) et post-traitement (vérification de la réponse).3. **Modular RAG** : Architecture flexible où chaque composant (retriever, reader, generator) peut être remplacé indépendamment.

## 2.2 — Architecture et pipeline RAGLe pipeline complet d'un système RAG se décompose en **6 étapes** :```┌─────────────────────────────────────────────────────────────┐│                    PIPELINE RAG                             ││                                                             ││  [Corpus de documents]                                      ││       │                                                     ││       ▼                                                     ││  ① Encodage des documents → Embeddings (vecteurs)           ││       │                                                     ││       ▼                                                     ││  ② Indexation dans une base vectorielle (FAISS)             ││                                                             ││  [Question utilisateur]                                     ││       │                                                     ││       ▼                                                     ││  ③ Encodage de la requête → Embedding de la question        ││       │                                                     ││       ▼                                                     ││  ④ Recherche des top-k documents pertinents (FAISS)         ││       │                                                     ││       ▼                                                     ││  ⑤ Construction du prompt augmenté (contexte + question)    ││       │                                                     ││       ▼                                                     ││  ⑥ Génération de la réponse par le LLM                      ││       │                                                     ││       ▼                                                     ││  [Réponse finale]                                           │└─────────────────────────────────────────────────────────────┘```**Détail des étapes :**1. **Encodage des documents** : Chaque document (ou chunk) est transformé en un vecteur dense à l'aide d'un modèle d'embeddings (ex: `sentence-transformers`).2. **Indexation** : Les vecteurs sont stockés dans un index FAISS pour permettre une recherche rapide par similarité.3. **Encodage de la requête** : La question de l'utilisateur est transformée en vecteur avec le même modèle d'embeddings.4. **Recherche top-k** : FAISS compare le vecteur de la question avec les vecteurs des documents et retourne les k plus proches.5. **Construction du prompt** : Les documents récupérés sont insérés dans un prompt structuré avec la question.6. **Génération** : Le LLM génère une réponse en se basant sur le contexte fourni.

## 2.3 — Implémentation du système RAG

In [ ]:
# Installation des bibliothèques nécessaires pour le RAG!pip install -q sentence-transformers faiss-cpu transformers accelerate

### 2.3.1 — Importation des bibliothèques

In [ ]:
import numpy as npimport faissimport torchfrom sentence_transformers import SentenceTransformerfrom transformers import pipeline

### 2.3.2 — Définition du corpus de documentsDans un système RAG, le corpus représente la **base de connaissances externe** que le modèle consulte pour répondre aux questions.Pour cette implémentation, nous utilisons un **dataset simplifié** composé de paragraphes couvrant des thématiques liées au NLP, aux Transformers et aux systèmes RAG.Chaque élément du corpus constitue un **chunk** (morceau de texte) directement utilisable pour la recherche vectorielle.

In [ ]:
# ============================================================# CORPUS : Dataset simplifié pour le système RAG# ============================================================documents = [    # --- Transformers ---    """L'architecture Transformer a été introduite en 2017 dans l'article 'Attention is All You Need'par Vaswani et al. Elle repose sur un mécanisme d'attention appelé self-attention, qui permetau modèle de pondérer l'importance de chaque mot par rapport aux autres mots de la séquence.Contrairement aux réseaux récurrents (RNN, LSTM), les Transformers traitent tous les motsen parallèle, ce qui accélère considérablement l'entraînement.""",    """Le mécanisme d'attention multi-têtes (multi-head attention) est un composant clé desTransformers. Il permet au modèle d'apprendre simultanément différentes représentationsde la relation entre les mots. Chaque tête d'attention capture un aspect différent ducontexte linguistique. Les résultats de toutes les têtes sont ensuite concaténés etprojetés pour former la représentation finale.""",    """BERT (Bidirectional Encoder Representations from Transformers) est un modèle pré-entraînédéveloppé par Google en 2018. Il utilise uniquement l'encodeur du Transformer et estentraîné de manière bidirectionnelle : il lit le texte dans les deux sens simultanément.BERT a révolutionné le NLP en établissant de nouveaux records sur de nombreuses tâchescomme la classification de texte, le question answering et la reconnaissance d'entités.""",    """GPT (Generative Pre-trained Transformer) est une famille de modèles développée par OpenAI.Contrairement à BERT qui utilise l'encodeur, GPT utilise le décodeur du Transformer.Il est entraîné de manière auto-régressive : il prédit le prochain mot à partir des motsprécédents. GPT-3 et GPT-4 sont des modèles très grands (175 milliards et plus de paramètres)capables de générer du texte cohérent et de réaliser de nombreuses tâches en zero-shot.""",    # --- NLP Tasks ---    """L'analyse de sentiment est une tâche NLP qui consiste à déterminer la polarité d'un texte :positif, négatif ou neutre. Elle est largement utilisée pour analyser les avis clients,les commentaires sur les réseaux sociaux et les critiques de produits. Les modèles Transformerscomme RoBERTa et XLM-RoBERTa obtiennent d'excellents résultats sur cette tâche, y comprisdans un contexte multilingue.""",    """Le Question Answering (QA) extractif consiste à trouver la réponse à une questiondirectement dans un texte source appelé contexte. Le modèle identifie la position de débutet de fin de la réponse dans le texte. Des modèles comme BERT, RoBERTa et XLM-RoBERTasont fréquemment utilisés pour cette tâche. Le dataset SQuAD (Stanford Question AnsweringDataset) est la référence pour évaluer les modèles de QA.""",    """Le résumé automatique (text summarization) consiste à produire une version condenséed'un texte long tout en conservant les informations essentielles. Il existe deux approches :le résumé extractif, qui sélectionne les phrases les plus importantes du texte original,et le résumé abstractif, qui génère de nouvelles phrases. Les modèles comme BART, T5 etmBART sont très performants pour le résumé abstractif.""",    # --- RAG ---    """Le RAG (Retrieval-Augmented Generation) est une approche qui combine la recherched'information avec la génération de texte. Introduit par Lewis et al. en 2020, le RAGpermet à un modèle de langage d'accéder à une base de connaissances externe avant degénérer sa réponse. Cela réduit les hallucinations et permet au modèle de fournir desréponses basées sur des données factuelles et vérifiables.""",    """Le pipeline RAG comprend plusieurs étapes : d'abord, les documents du corpus sontencodés en vecteurs (embeddings) à l'aide d'un modèle comme Sentence-BERT. Ces vecteurssont indexés dans une base vectorielle comme FAISS. Quand l'utilisateur pose une question,celle-ci est également encodée en vecteur, puis les documents les plus similaires sontrécupérés. Enfin, ces documents sont insérés dans le prompt du modèle génératif.""",    """Les embeddings sont des représentations vectorielles denses qui capturent le senssémantique d'un texte. Deux textes ayant un sens similaire auront des vecteurs prochesdans l'espace vectoriel. Les modèles Sentence-Transformers, comme 'paraphrase-multilingual-MiniLM',sont spécifiquement entraînés pour produire des embeddings de haute qualité pour larecherche sémantique et le clustering de textes.""",    """FAISS (Facebook AI Similarity Search) est une bibliothèque développée par Meta pourla recherche efficace de similarité dans de grands ensembles de vecteurs. FAISS supporteplusieurs types d'index : IndexFlatL2 pour la recherche exacte par distance euclidienne,IndexIVFFlat pour une recherche approximative plus rapide, et IndexHNSW pour les graphesde voisinage. Dans un système RAG, FAISS permet de retrouver rapidement les documentsles plus pertinents parmi des millions de vecteurs.""",    # --- Fine-tuning vs RAG ---    """Le fine-tuning consiste à ré-entraîner un modèle pré-entraîné sur un jeu de donnéesspécifique pour l'adapter à une tâche particulière. Cette approche nécessite des donnéesannotées, des ressources GPU importantes et du temps d'entraînement. L'avantage est quele modèle internalise les connaissances spécialisées. L'inconvénient est que les donnéesdoivent être mises à jour par un nouveau cycle d'entraînement.""",    """Le RAG présente plusieurs avantages par rapport au fine-tuning : la mise à jour desconnaissances est instantanée (il suffit de modifier le corpus), les réponses sonttraçables (on peut identifier les sources utilisées), et le coût est moindre car il nenécessite pas de ré-entraîner le modèle. En revanche, le RAG dépend de la qualité dumoteur de recherche et de la pertinence des documents récupérés.""",    """Le prompt engineering est une approche qui consiste à formuler des instructionsprécises et optimisées pour guider le modèle de langage vers la réponse souhaitée.Contrairement au fine-tuning, elle ne modifie pas les poids du modèle. Contrairementau RAG, elle n'utilise pas de base de connaissances externe. Le prompt engineering estlimité par la taille de la fenêtre de contexte du modèle et par ses connaissances internes.""",    # --- Applications ---    """Les systèmes RAG sont utilisés dans de nombreuses applications : les chatbotsdocumentaires qui répondent à des questions sur des documents internes d'entreprise,les assistants de recherche juridique qui analysent des textes de loi, les systèmesde support technique qui consultent des manuels, et les outils éducatifs qui répondentà des questions sur des cours. Le RAG est particulièrement utile quand les donnéeschangent fréquemment ou sont spécifiques à un domaine.""",]print(f"Nombre de documents dans le corpus : {len(documents)}")print(f"\nAperçu des 3 premiers documents :")for i, doc in enumerate(documents[:3]):    print(f"\n--- Document {i+1} ---")    print(doc[:200] + "...")

### 2.3.3 — Préparation du corpusNotre corpus est déjà structuré sous forme de **chunks** (paragraphes thématiques).Dans un cas réel avec des documents longs (PDF, articles), il faudrait :1. Extraire le texte brut du document.2. Découper le texte en chunks de taille fixe avec chevauchement.Ici, chaque document du dataset constitue directement un chunk prêt à être encodé en embedding.

In [ ]:
# Les documents du corpus sont directement utilisés comme chunksdocuments_corpus = documentsprint(f"Nombre de chunks prêts pour l'encodage : {len(documents_corpus)}")print(f"Longueur moyenne d'un chunk : {np.mean([len(d.split()) for d in documents_corpus]):.0f} mots")

### 2.3.4 — Génération des embeddingsUn **embedding** est une représentation vectorielle d'un texte.Deux textes de sens proche auront des vecteurs proches dans l'espace vectoriel.Nous utilisons le modèle `paraphrase-multilingual-MiniLM-L12-v2` qui produit des embeddings multilingues de 384 dimensions.

In [ ]:
# Chargement du modèle d'embeddingsmodele_embeddings = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"print("Chargement du modèle d'embeddings...")embedder = SentenceTransformer(modele_embeddings)print("Modèle chargé :", modele_embeddings)

### 2.3.5 — Encodage des documentsChaque document du corpus est transformé en un vecteur dense de 384 dimensions.

In [ ]:
# Encodage des documents en embeddingsprint("Encodage des documents en embeddings...")document_embeddings = embedder.encode(    documents_corpus,    convert_to_numpy=True,    show_progress_bar=True)document_embeddings = document_embeddings.astype("float32")print("Forme de la matrice des embeddings :")print(document_embeddings.shape)

### 2.3.6 — Indexation avec FAISSFAISS permet de rechercher rapidement les documents dont les embeddings sont les plus proches de l'embedding de la question utilisateur.

In [ ]:
dimension = document_embeddings.shape[1]index = faiss.IndexFlatL2(dimension)index.add(document_embeddings)print("Indexation terminée.")print("Nombre de vecteurs dans FAISS :", index.ntotal)print("Dimension des vecteurs :", dimension)

### 2.3.7 — Recherche des documents pertinentsLorsqu'un utilisateur pose une question, nous transformons cette question en embedding.FAISS recherche les documents les plus proches.

In [ ]:
def rechercher_documents_pertinents(requete, k=3):    requete_embedding = embedder.encode(        [requete],        convert_to_numpy=True    ).astype("float32")    distances, indices = index.search(requete_embedding, k)    resultats = []    for rang, idx in enumerate(indices[0]):        resultats.append({            "rang": rang + 1,            "distance": distances[0][rang],            "texte": documents_corpus[idx]        })    return resultats

**Test de recherche :**

In [ ]:
question_test = "Qu'est-ce qu'un système RAG ?"resultats = rechercher_documents_pertinents(question_test, k=3)print("Question :", question_test)print("\nDocuments récupérés :")for res in resultats:    print(f"\n--- Document {res['rang']} | Distance : {res['distance']:.4f} ---")    print(res["texte"][:700])

### 2.3.8 — Génération de réponse avec un prompt augmentéLe prompt contient :1. Le contexte récupéré depuis le corpus.2. La question de l'utilisateur.3. Une instruction demandant au modèle de répondre uniquement à partir du contexte.

In [ ]:
device = 0 if torch.cuda.is_available() else -1print("Chargement du modèle génératif...")llm_pipeline = pipeline(    "text2text-generation",    model="google/flan-t5-small",    device=device)print("Modèle génératif chargé.")

### Fonction RAG complète

In [ ]:
def systeme_rag(question, k=3):    # 1. Recherche des documents pertinents    resultats = rechercher_documents_pertinents(question, k=k)    # 2. Construction du contexte    contexte = "\n\n".join([res["texte"] for res in resultats])    # 3. Construction du prompt augmenté    prompt = f"""Réponds à la question en te basant uniquement sur le contexte fourni.Contexte :{contexte}Question :{question}Réponse :"""    # 4. Génération de la réponse    reponse = llm_pipeline(        prompt,        max_new_tokens=150,        truncation=True    )[0]["generated_text"]    return reponse, resultats, prompt

## 2.4 — Démonstration et expérimentation### Test du système RAG avec plusieurs questionsNous testons le système avec **5 questions différentes** pour évaluer sa capacité à répondre à des questions variées sur le corpus.

In [ ]:
questions_test = [    "Qu'est-ce qu'un système RAG ?",    "Quelle est la différence entre le fine-tuning et le RAG ?",    "Comment fonctionne le mécanisme d'attention dans les Transformers ?",    "Quels sont les avantages du RAG par rapport à un LLM classique ?",    "Comment les embeddings sont-ils utilisés dans un système RAG ?",]print("=" * 70)print("TEST DU SYSTÈME RAG AVEC PLUSIEURS QUESTIONS")print("=" * 70)for i, question in enumerate(questions_test, 1):    reponse, sources, _ = systeme_rag(question, k=3)    print(f"\n{'─' * 60}")    print(f"Question {i} : {question}")    print(f"{'─' * 60}")    print(f"Réponse RAG : {reponse}")    print(f"\nSources utilisées :")    for s in sources:        print(f"  • Doc {s['rang']} (distance: {s['distance']:.4f}) : {s['texte'][:120]}...")

### 2.4.1 — Comparaison des réponses avec et sans RAGSans RAG, le modèle répond uniquement avec ses connaissances internes.Avec RAG, le modèle reçoit un contexte extrait du corpus avant de générer la réponse.Cette comparaison permet de démontrer l'apport du RAG.

In [ ]:
def repondre_sans_rag(question):    prompt = f"""Réponds à la question suivante :Question :{question}Réponse :"""    reponse = llm_pipeline(        prompt,        max_new_tokens=120,        truncation=True    )[0]["generated_text"]    return reponse

In [ ]:
questions_comparaison = [    "Qu'est-ce qu'un système RAG ?",    "Quelle est la différence entre le fine-tuning et le RAG ?",    "Comment fonctionne le mécanisme d'attention dans les Transformers ?",    "Quels sont les avantages du RAG par rapport à un LLM classique ?",    "Comment les embeddings sont-ils utilisés dans un système RAG ?",]print("=" * 80)print("COMPARAISON DES RÉPONSES : AVEC vs SANS RAG")print("=" * 80)for i, question in enumerate(questions_comparaison, 1):    reponse_sans = repondre_sans_rag(question)    reponse_avec, sources, _ = systeme_rag(question, k=3)    print(f"\n{'━' * 80}")    print(f"  Question {i} : {question}")    print(f"{'━' * 80}")    print(f"\n  📭 Réponse SANS RAG :")    print(f"     {reponse_sans}")    print(f"\n  📬 Réponse AVEC RAG :")    print(f"     {reponse_avec}")    print(f"\n  📎 Sources RAG :")    for s in sources:        print(f"     • Doc {s['rang']} (dist: {s['distance']:.4f})")print(f"\n{'━' * 80}")print("FIN DE LA COMPARAISON")print(f"{'━' * 80}")

# Conclusion## Résumé de la Partie 1Dans la première partie, nous avons exploré les modèles Transformers pré-entraînés via la bibliothèque HuggingFace :- **Tokenizer BERT** : Nous avons observé comment un texte brut est découpé en tokens puis en identifiants numériques.- **Analyse de sentiment** : Le modèle `xlm-roberta` classifie correctement les opinions positives et négatives.- **Classification zero-shot** : Le modèle `BART` permet de classer un texte sans entraînement spécifique.- **Question Answering** : Le modèle extrait des réponses précises à partir d'un contexte donné.- **Résumé automatique** : Le modèle condense un texte long en conservant les idées principales.## Résumé de la Partie 2Dans la deuxième partie, nous avons :1. Présenté l'**état de l'art** des systèmes RAG, leurs avantages par rapport au fine-tuning et au prompt engineering.2. Formalisé l'**architecture complète** du pipeline RAG en 6 étapes.3. **Implémenté** un système RAG fonctionnel utilisant :   - Un **dataset simplifié** comme corpus de connaissances (14 documents thématiques)   - `sentence-transformers` pour la génération d'embeddings   - `FAISS` pour l'indexation et la recherche vectorielle   - `Flan-T5` pour la génération de réponses4. **Comparé** les réponses avec et sans RAG sur 5 questions différentes, montrant que le RAG permet de fournir des réponses plus précises et contextualisées grâce à l'accès aux documents externes.## Observations- Le RAG améliore significativement la pertinence des réponses lorsque la question porte sur le contenu du corpus.- Sans RAG, le modèle se limite à ses connaissances générales, souvent insuffisantes pour des questions spécifiques.- La qualité des embeddings et de l'index vectoriel est cruciale pour la performance du système.- L'utilisation d'un dataset simplifié permet de tester le concept sans dépendance à des fichiers externes.